**Voice-to-Grasp - Experiment 4, analysis of one participant's run** of `continual_batch_learning.py`
(training and test phase). Outputs were cleared for the release.

# VoiceToGrasp — Continual Learning Experiment Analysis

This notebook analyses the two-phase experiment produced by `continual_batch_learning.py`:

| Phase | Rows | Purpose |
|---|---|---|
| **Phase 1 — Continual Learning** | 1 – 140 | Evaluate → update on errors |
| **Phase 2 — Held-out Test** | 141 – 210 | Pure inference, model frozen |

**Outputs read**
- `experiment_results/<timestamp>/results_training_phase.csv`
- `experiment_results/<timestamp>/results_test_phase.csv`
- `experiment_results/<timestamp>/experiment_summary.json`
- `Continual_Learning_Experiments/<user_id>/log.csv`  *(trainer log)*


## 0  ·  Configuration — set your paths here

In [ ]:
import os, json, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.metrics import (
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_auc_score
)
from sklearn.preprocessing import label_binarize
from scipy.stats import wilcoxon
from statsmodels.stats.proportion import proportion_confint

sns.set_theme(style="whitegrid", palette="tab10")
plt.rcParams.update({"figure.dpi": 120, "axes.titlesize": 13, "axes.labelsize": 11})

# ── EDIT THESE ──────────────────────────────────────────────────────
RESULTS_DIR  = r"D:\PythonEnvs\nlpglove\Continual_Learning_Experiments\UserStudy_code_analysis\experiment_results"          # folder created by the trainer
USER_ID      = "P2"  # your user ID (e.g., "P3", "P4", "P2", "PilotA")
EXPERIMENTS_ROOT = "Continual_Learning_Experiments"
# ────────────────────────────────────────────────────────────────────

LABEL_ORDER = ["grasp", "open_hand", "gentle_grasp",
               "tight_grasp", "pinch_grasp", "tighten", "loosen"]

# Auto-pick the most recent experiment folder
runs = sorted(glob.glob(os.path.join(RESULTS_DIR, USER_ID, "*")))
if not runs:
    raise FileNotFoundError(
        f"No experiment folders found in '{RESULTS_DIR}'. "
        "Run continual_batch_learning.py first."
    )
LATEST = runs[-1]
print(f"Loading experiment: {LATEST}")

## 1  ·  Load Data

In [ ]:
# Phase CSVs
train_df = pd.read_csv(os.path.join(LATEST, "results_training_phase.csv"))
test_df  = pd.read_csv(os.path.join(LATEST, "results_test_phase.csv"))

# JSON summary
with open(os.path.join(LATEST, "experiment_summary.json")) as f:
    summary = json.load(f)

# Trainer log (optional — may not exist if path differs)
log_path = os.path.join(EXPERIMENTS_ROOT, USER_ID, "log.csv")
if os.path.exists(log_path):
    log_df = pd.read_csv(log_path, encoding="latin1")
    log_df.replace("", np.nan, inplace=True)
    log_df["is_correct"]   = log_df["correct_label"].isna()
    log_df["confidence"]   = pd.to_numeric(log_df["confidence"],   errors="coerce")
    log_df["loss"]         = pd.to_numeric(log_df["loss"],         errors="coerce")
    log_df["update_number"]= pd.to_numeric(log_df["update_number"],errors="coerce")
    log_df["buffer_size"]  = pd.to_numeric(log_df["buffer_size"],  errors="coerce")
    HAS_LOG = True
else:
    print(f"Trainer log not found at {log_path} — log-based cells will be skipped.")
    HAS_LOG = False

print(f"Training samples : {len(train_df)}")
print(f"Test samples     : {len(test_df)}")
train_df.head(3)

## 2  ·  Dataset Overview

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, df, title in zip(
    axes,
    [train_df, test_df],
    ["Phase 1 — Training (rows 1-140)", "Phase 2 — Test (rows 141-210)"]
):
    counts = df["true_label"].value_counts().reindex(LABEL_ORDER).fillna(0)
    bars = ax.bar(counts.index, counts.values, color=sns.color_palette("tab10", len(counts)))
    ax.set_title(title)
    ax.set_ylabel("Count")
    ax.set_xlabel("Grasp Type")
    ax.tick_params(axis="x", rotation=35)
    for b in bars:
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.3,
                int(b.get_height()), ha="center", fontsize=8)

plt.suptitle("Label Distribution per Phase", fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 3  ·  High-Level Accuracy Summary

In [ ]:
train_acc = train_df["correct"].mean()
test_acc  = test_df["correct"].mean()
train_updates = int(train_df["loss"].notna().sum())

print("=" * 45)
print(f"  Phase 1 Accuracy  (training)  : {train_acc:.4f}")
print(f"  Phase 2 Accuracy  (test)      : {test_acc:.4f}")
print(f"  Model updates triggered       : {train_updates}")
print(f"  Update rate                   : {train_updates/len(train_df):.4f}")
print("=" * 45)

# Bar chart
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(["Phase 1\nContinual Learning", "Phase 2\nHeld-out Test"],
              [train_acc, test_acc],
              color=["steelblue", "darkorange"], width=0.45, edgecolor="white")
for b, v in zip(bars, [train_acc, test_acc]):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.01,
            f"{v:.3f}", ha="center", fontweight="bold", fontsize=12)
ax.set_ylim(0, 1.15)
ax.set_ylabel("Accuracy")
ax.set_title("Overall Accuracy by Phase")
ax.axhline(0.5, color="grey", linestyle=":", linewidth=0.8, label="Chance (0.50)")
ax.legend()
plt.tight_layout()
plt.show()

## 4  ·  Learning Curve (Phase 1)

In [ ]:
train_df["running_acc"] = train_df["correct"].expanding().mean()

# Rolling window accuracy (smoothed)
win = 20
train_df["rolling_acc"] = train_df["correct"].rolling(win, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(train_df["index"], train_df["running_acc"],
        color="steelblue", linewidth=1.5, label="Cumulative accuracy")
ax.plot(train_df["index"], train_df["rolling_acc"],
        color="darkorange", linewidth=1.2, linestyle="--",
        label=f"Rolling accuracy (w={win})")
ax.axhline(train_acc, color="tomato", linestyle=":", linewidth=1,
           label=f"Final={train_acc:.3f}")
ax.set_xlabel("Sample #")
ax.set_ylabel("Accuracy")
ax.set_title("Phase 1 — Running Accuracy During Continual Learning")
ax.set_ylim(0, 1.05)
ax.legend()
plt.tight_layout()
plt.show()

## 5  ·  Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for ax, df, title in zip(
    axes,
    [train_df, test_df],
    ["Phase 1 — Continual Learning", "Phase 2 — Held-out Test"]
):
    y_true = df["true_label"]
    y_pred = df["pred_label"]
    present = [l for l in LABEL_ORDER if l in set(y_true) | set(y_pred)]
    cm = confusion_matrix(y_true, y_pred, labels=present)
    disp = ConfusionMatrixDisplay(cm, display_labels=present)
    disp.plot(ax=ax, colorbar=False, cmap="Blues", xticks_rotation=40)
    ax.set_title(title)

plt.suptitle("Confusion Matrices", fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 6  ·  Per-Class Metrics

In [ ]:
def per_class_report(df, phase_name):
    y_true = df["true_label"]
    y_pred = df["pred_label"]
    present = [l for l in LABEL_ORDER if l in set(y_true)]
    rpt = classification_report(y_true, y_pred, labels=present,
                                zero_division=0, output_dict=True)
    rows = []
    for label in present:
        rows.append({
            "label":     label,
            "precision": rpt[label]["precision"],
            "recall":    rpt[label]["recall"],
            "f1-score":  rpt[label]["f1-score"],
            "support":   int(rpt[label]["support"]),
        })
    rdf = pd.DataFrame(rows).set_index("label")
    print(f"\n{'='*50}")
    print(f"  {phase_name}")
    print(f"{'='*50}")
    print(classification_report(y_true, y_pred, labels=present, zero_division=0))
    return rdf

rpt_train = per_class_report(train_df, "Phase 1 — Continual Learning")
rpt_test  = per_class_report(test_df,  "Phase 2 — Held-out Test")

In [ ]:
# Side-by-side F1 comparison
fig, ax = plt.subplots(figsize=(11, 4))
x = np.arange(len(rpt_train))
w = 0.35
ax.bar(x - w/2, rpt_train["f1-score"], w, label="Phase 1", color="steelblue")
ax.bar(x + w/2, rpt_test.reindex(rpt_train.index)["f1-score"].fillna(0),
       w, label="Phase 2", color="darkorange")
ax.set_xticks(x)
ax.set_xticklabels(rpt_train.index, rotation=35, ha="right")
ax.set_ylabel("F1-Score")
ax.set_ylim(0, 1.15)
ax.set_title("Per-Class F1-Score: Phase 1 vs Phase 2")
ax.legend()
ax.grid(axis="y", alpha=0.4)
plt.tight_layout()
plt.show()

## 7  ·  Confidence Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)

for ax, df, title, color in zip(
    axes,
    [train_df, test_df],
    ["Phase 1 — Training", "Phase 2 — Test"],
    ["steelblue", "darkorange"]
):
    correct   = df[df["correct"]]["confidence"]
    incorrect = df[~df["correct"]]["confidence"]
    ax.hist(correct,   bins=15, range=(0,1), alpha=0.7, color="green",  label=f"Correct  (n={len(correct)})")
    ax.hist(incorrect, bins=15, range=(0,1), alpha=0.7, color="tomato", label=f"Incorrect (n={len(incorrect)})")
    ax.axvline(df["confidence"].mean(), color="black", linestyle="--",
               label=f"Mean={df['confidence'].mean():.3f}")
    ax.set_xlabel("Confidence")
    ax.set_ylabel("Count")
    ax.set_title(title)
    ax.legend(fontsize=8)

plt.suptitle("Confidence Distribution — Correct vs Incorrect", fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Confidence box-plot per class — test phase
fig, ax = plt.subplots(figsize=(11, 4))
order = [l for l in LABEL_ORDER if l in test_df["true_label"].unique()]
sns.boxplot(data=test_df, x="true_label", y="confidence",
            order=order, palette="tab10", ax=ax)
ax.set_title("Phase 2 — Confidence per True Class")
ax.set_xlabel("True Grasp Type")
ax.set_ylabel("Confidence")
ax.tick_params(axis="x", rotation=35)
plt.tight_layout()
plt.show()

## 8  ·  Calibration (Expected Calibration Error)

In [ ]:
def ece(confidences, correct, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    ece_val = 0
    bin_accs, bin_confs, bin_sizes = [], [], []
    for i in range(n_bins):
        mask = (confidences >= bins[i]) & (confidences < bins[i+1])
        if mask.sum() > 0:
            ba = correct[mask].mean()
            bc = confidences[mask].mean()
            ece_val += abs(ba - bc) * mask.sum() / len(confidences)
            bin_accs.append(ba); bin_confs.append(bc); bin_sizes.append(mask.sum())
    return ece_val, bin_accs, bin_confs, bin_sizes

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, df, title in zip(
    axes,
    [train_df, test_df],
    ["Phase 1 — Training", "Phase 2 — Test"]
):
    conf = df["confidence"].values
    corr = df["correct"].astype(float).values
    ece_val, bin_accs, bin_confs, bin_sizes = ece(conf, corr)

    ax.plot([0,1], [0,1], "k--", linewidth=1, label="Perfect calibration")
    sc = ax.scatter(bin_confs, bin_accs, c=bin_sizes, cmap="YlOrRd",
                    s=80, zorder=5, edgecolors="grey", linewidths=0.5)
    plt.colorbar(sc, ax=ax, label="Bin size")
    ax.set_xlim(0,1); ax.set_ylim(0,1)
    ax.set_xlabel("Mean Confidence"); ax.set_ylabel("Fraction Correct")
    ax.set_title(f"{title}\nECE = {ece_val:.4f}")
    ax.legend(fontsize=8)

plt.suptitle("Reliability / Calibration Diagram", fontweight="bold")
plt.tight_layout()
plt.show()

## 9  ·  Loss & Update Analysis (Phase 1)

In [ ]:
update_df = train_df.dropna(subset=["loss"]).copy()
update_df["loss"] = pd.to_numeric(update_df["loss"], errors="coerce")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Loss over updates
ax = axes[0]
ax.plot(update_df["update_num"], update_df["loss"],
        color="tomato", linewidth=1.2, marker="o", markersize=3)
# Rolling mean
roll = update_df["loss"].rolling(5, min_periods=1).mean()
ax.plot(update_df["update_num"], roll, color="darkred", linewidth=1.5,
        linestyle="--", label="Rolling mean (w=5)")
ax.set_xlabel("Update #"); ax.set_ylabel("Loss")
ax.set_title("Training Loss per Update")
ax.legend()

# Update timing strip
ax = axes[1]
updated = train_df["loss"].notna()
colors  = ["tomato" if u else "steelblue" for u in updated]
ax.bar(train_df["index"], [1]*len(train_df), color=colors, width=1, edgecolor="none")
up_patch = mpatches.Patch(color="tomato",    label=f"Updated ({updated.sum()})")
ok_patch = mpatches.Patch(color="steelblue", label=f"Correct ({(~updated).sum()})")
ax.legend(handles=[up_patch, ok_patch], fontsize=9)
ax.set_yticks([]); ax.set_xlabel("Sample index")
ax.set_title("When Model Updates Occurred")

plt.tight_layout()
plt.show()

print(f"Total updates : {len(update_df)}")
print(f"Mean loss     : {update_df['loss'].mean():.4f}")
print(f"Min  loss     : {update_df['loss'].min():.4f}")
print(f"Max  loss     : {update_df['loss'].max():.4f}")

## 10  ·  Which Classes Were Hardest to Learn? (Phase 1)

In [ ]:
hard = train_df[~train_df["correct"]].copy()
easy = train_df[ train_df["correct"]].copy()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Errors per class
ax = axes[0]
err_counts = hard["true_label"].value_counts().reindex(LABEL_ORDER).fillna(0)
ax.bar(err_counts.index, err_counts.values, color="tomato", edgecolor="white")
ax.set_title("Errors per True Class (Phase 1)")
ax.set_ylabel("# Errors (updates)")
ax.tick_params(axis="x", rotation=35)

# Error rate per class
ax = axes[1]
total_per_class = train_df["true_label"].value_counts().reindex(LABEL_ORDER).fillna(0)
err_rate = (err_counts / total_per_class).fillna(0)
bars = ax.bar(err_rate.index, err_rate.values,
              color=[plt.cm.RdYlGn(1 - v) for v in err_rate.values])
for b, v in zip(bars, err_rate.values):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.01,
            f"{v:.0%}", ha="center", fontsize=8)
ax.set_ylim(0, 1.1)
ax.set_title("Error Rate per Class (Phase 1)")
ax.set_ylabel("Error Rate")
ax.tick_params(axis="x", rotation=35)

plt.tight_layout()
plt.show()

## 11  ·  Plasticity vs Stability

In [ ]:
# Split Phase 1 into early / late halves
mid = len(train_df) // 2
early = train_df.iloc[:mid]
late  = train_df.iloc[mid:]

# Per-class accuracy in early vs late
def class_acc(df):
    return df.groupby("true_label")["correct"].mean().reindex(LABEL_ORDER)

early_acc = class_acc(early)
late_acc  = class_acc(late)
delta     = late_acc - early_acc

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
x = np.arange(len(LABEL_ORDER))
w = 0.35
ax.bar(x - w/2, early_acc.fillna(0), w, label=f"Early (1–{mid})", color="cornflowerblue")
ax.bar(x + w/2, late_acc.fillna(0),  w, label=f"Late ({mid+1}–140)", color="steelblue")
ax.set_xticks(x); ax.set_xticklabels(LABEL_ORDER, rotation=35, ha="right")
ax.set_ylim(0, 1.15); ax.set_ylabel("Accuracy")
ax.set_title("Per-Class Accuracy: Early vs Late (Phase 1)")
ax.legend()

ax = axes[1]
colors_delta = ["green" if v >= 0 else "tomato" for v in delta.fillna(0)]
ax.bar(LABEL_ORDER, delta.fillna(0), color=colors_delta, edgecolor="white")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylabel("Accuracy Change (Late − Early)")
ax.set_title("Learning Gain per Class (Phase 1)")
ax.tick_params(axis="x", rotation=35)
for i, v in enumerate(delta.fillna(0)):
    ax.text(i, v + (0.01 if v >= 0 else -0.04),
            f"{v:+.2f}", ha="center", fontsize=8)

plt.tight_layout()
plt.show()

print(f"Plasticity (accuracy last 20 samples) : {train_df.tail(20)['correct'].mean():.4f}")
print(f"Stability  (accuracy first 20 samples): {train_df.head(20)['correct'].mean():.4f}")

## 12  ·  Generalisation: Phase 1 → Phase 2

In [ ]:
# Per-class accuracy comparison across both phases
train_acc_cls = train_df.groupby("true_label")["correct"].mean().reindex(LABEL_ORDER)
test_acc_cls  = test_df.groupby("true_label")["correct"].mean().reindex(LABEL_ORDER)

fig, ax = plt.subplots(figsize=(11, 4))
x = np.arange(len(LABEL_ORDER))
w = 0.35
ax.bar(x - w/2, train_acc_cls.fillna(0), w, label="Phase 1", color="steelblue")
ax.bar(x + w/2, test_acc_cls.fillna(0),  w, label="Phase 2", color="darkorange")
ax.set_xticks(x); ax.set_xticklabels(LABEL_ORDER, rotation=35, ha="right")
ax.set_ylim(0, 1.15); ax.set_ylabel("Accuracy")
ax.set_title("Per-Class Accuracy: Phase 1 (training) vs Phase 2 (test)")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

# Wilcoxon signed-rank test between phase accuracies per class
shared = [l for l in LABEL_ORDER
          if l in train_df["true_label"].values and l in test_df["true_label"].values]
p1_accs = [train_df[train_df["true_label"]==l]["correct"].mean() for l in shared]
p2_accs = [test_df[test_df["true_label"]==l]["correct"].mean()   for l in shared]

if len(shared) >= 3:
    stat, pval = wilcoxon(p1_accs, p2_accs, zero_method="zsplit")
    print(f"Wilcoxon signed-rank test (Phase 1 vs Phase 2 per-class accuracy):")
    print(f"  statistic = {stat:.3f},  p = {pval:.4f}")
    print(f"  {'Significant difference' if pval < 0.05 else 'No significant difference'} at α=0.05")

## 13  ·  Wilson Confidence Intervals

In [ ]:
from statsmodels.stats.proportion import proportion_confint

def ci_row(df, label):
    n_correct = df["correct"].sum()
    n_total   = len(df)
    lo, hi = proportion_confint(n_correct, n_total, method="wilson")
    return {
        "Phase": label,
        "Accuracy": n_correct / n_total,
        "95% CI lower": lo,
        "95% CI upper": hi,
        "n": n_total,
    }

ci_df = pd.DataFrame([
    ci_row(train_df, "Phase 1 — Training"),
    ci_row(test_df,  "Phase 2 — Test"),
])
display(ci_df.set_index("Phase").style.format("{:.4f}"))

fig, ax = plt.subplots(figsize=(6, 3))
for i, row in ci_df.iterrows():
    color = "steelblue" if "Phase 1" in row["Phase"] else "darkorange"
    ax.errorbar(
        x=row["Accuracy"],
        y=row["Phase"],
        xerr=[[row["Accuracy"] - row["95% CI lower"]],
              [row["95% CI upper"] - row["Accuracy"]]],
        fmt="o", color=color, capsize=6, markersize=8
    )
ax.set_xlabel("Accuracy")
ax.set_title("Accuracy with 95% Wilson CI")
ax.set_xlim(0, 1)
ax.axvline(0.5, color="grey", linestyle=":", linewidth=0.8)
plt.tight_layout()
plt.show()

## 14  ·  Replay Buffer Growth (from Trainer Log)

In [ ]:
if HAS_LOG:
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot(log_df.index, log_df["buffer_size"], color="steelblue", linewidth=1.2)
    ax.fill_between(log_df.index, log_df["buffer_size"], alpha=0.15, color="steelblue")
    ax.set_xlabel("Sample Index")
    ax.set_ylabel("Buffer Size")
    ax.set_title("Replay Buffer Growth Over Training")
    plt.tight_layout()
    plt.show()
else:
    print("Trainer log not available — skipping buffer plot.")

## 15  ·  Misclassification Patterns

In [ ]:
def misclass_heatmap(df, title):
    err = df[~df["correct"]]
    if len(err) == 0:
        print(f"{title}: no errors found.")
        return
    pivot = err.groupby(["true_label", "pred_label"]).size().unstack(fill_value=0)
    pivot = pivot.reindex(index=LABEL_ORDER, columns=LABEL_ORDER, fill_value=0)
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(pivot, annot=True, fmt="d", cmap="Oranges", ax=ax,
                linewidths=0.5, linecolor="white")
    ax.set_title(f"{title}\nRows = True Label, Columns = Predicted Label")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    plt.tight_layout()
    plt.show()

misclass_heatmap(train_df, "Phase 1 — Misclassification Patterns")
misclass_heatmap(test_df,  "Phase 2 — Misclassification Patterns")

## 16  ·  Summary Dashboard

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

# 1. Phase accuracy bar
ax = axes[0]
ax.bar(["Train", "Test"], [train_df["correct"].mean(), test_df["correct"].mean()],
       color=["steelblue", "darkorange"])
ax.set_ylim(0, 1.1); ax.set_ylabel("Accuracy"); ax.set_title("Overall Accuracy")

# 2. Learning curve
ax = axes[1]
ax.plot(train_df["index"], train_df["running_acc"], color="steelblue", linewidth=1.5)
ax.set_ylim(0, 1.05); ax.set_xlabel("Sample #"); ax.set_ylabel("Acc")
ax.set_title("Running Accuracy (Phase 1)")

# 3. Loss curve
ax = axes[2]
update_df2 = train_df.dropna(subset=["loss"]).copy()
update_df2["loss"] = pd.to_numeric(update_df2["loss"], errors="coerce")
ax.plot(update_df2["update_num"], update_df2["loss"], color="tomato", linewidth=1.2)
ax.set_xlabel("Update #"); ax.set_ylabel("Loss"); ax.set_title("Training Loss")

# 4. F1 comparison
ax = axes[3]
x = np.arange(len(rpt_train))
ax.bar(x - 0.2, rpt_train["f1-score"], 0.4, label="Phase 1", color="steelblue")
ax.bar(x + 0.2, rpt_test.reindex(rpt_train.index)["f1-score"].fillna(0),
       0.4, label="Phase 2", color="darkorange")
ax.set_xticks(x); ax.set_xticklabels(rpt_train.index, rotation=40, ha="right", fontsize=7)
ax.set_ylim(0, 1.15); ax.set_ylabel("F1"); ax.set_title("F1 per Class")
ax.legend(fontsize=8)

# 5. Confidence histogram
ax = axes[4]
ax.hist(train_df[train_df["correct"]]["confidence"],  bins=12, alpha=0.6,
        color="green", label="Correct")
ax.hist(train_df[~train_df["correct"]]["confidence"], bins=12, alpha=0.6,
        color="tomato", label="Incorrect")
ax.set_xlabel("Confidence"); ax.set_ylabel("Count")
ax.set_title("Confidence (Phase 1)")
ax.legend(fontsize=8)

# 6. Error rate per class (phase 1)
ax = axes[5]
err_counts2 = train_df[~train_df["correct"]]["true_label"].value_counts().reindex(LABEL_ORDER).fillna(0)
total2 = train_df["true_label"].value_counts().reindex(LABEL_ORDER).fillna(1)
err_rate2 = (err_counts2 / total2).fillna(0)
ax.bar(LABEL_ORDER, err_rate2, color=[plt.cm.RdYlGn(1 - v) for v in err_rate2])
ax.set_ylim(0, 1.1); ax.tick_params(axis="x", rotation=40, labelsize=7)
ax.set_ylabel("Error Rate"); ax.set_title("Error Rate per Class (Phase 1)")

plt.suptitle("VoiceToGrasp — Experiment Summary Dashboard",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()